# 01 — R2 playground

R2 is a **narrow** assistant: every request becomes one validated JSON *intent*,
then an allow-listed tool runs it. The model never executes anything itself.

```
speech/text → Qwen3-1.7B (MLX, KV prefix cache) → {"intent", "parameters"}
            → Pydantic validation → permission gate (+ confirmation) → tool → spoken reply
```
Run the cells top to bottom. The **chat widget** at the end is fully interactive.

In [1]:
# --- setup: make the repo importable and quiet logs ---
import os, sys, logging
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_OFFLINE", "1")  # models are already cached → fully local
logging.basicConfig(level=logging.WARNING)
print("repo:", ROOT)

repo: /Users/siavash/Projects/articles/narrow-ai-with-kv


In [2]:
from narrow_ai.r2 import R2
r2 = R2(workspace=ROOT)   # loads Qwen3 + KV cache (from disk if saved before)
r2.status()

{'name': 'R2',
 'state': 'idle',
 'backend': 'mlx',
 'model': 'mlx-community/Qwen3-1.7B-4bit',
 'workspace': '/Users/siavash/Projects/articles/narrow-ai-with-kv',
 'load_ms': 3835.4,
 'warmup_ms': 801.9,
 'kv_cache': {'mode': 'persistent',
  'source': 'disk',
  'prefix_tokens': 820,
  'build_ms': 0.0,
  'load_ms': 57.3,
  'hits': 0,
  'misses': 0,
  'mb': 117.4},
 'pending': None}

## Programmatic use

In [3]:
def show(resp):
    m = resp.metrics
    print(f"intent  : {resp.decision.intent.value}  {resp.decision.parameters}")
    print(f"speech  : {resp.spoken_text()}")
    print(f"latency : ttft {m.get('ttft_ms')} ms · total {m.get('think_ms')} ms · "
          f"{m.get('cached_tokens')} cached + {m.get('prompt_tokens')} new tokens")
    print("-" * 70)

for q in ["what can you do?", "find the notebooks", "which branch am I on",
          "show me AGENTS.md", "write me a haiku"]:
    print("you     :", q)
    show(r2.ask(q))

you     : what can you do?
intent  : general_help  {}
speech  : I can check git status and diffs, find and read files, open apps, and remember your preferences.
latency : ttft 66.7 ms · total 162.3 ms · 820 cached + 21 new tokens
----------------------------------------------------------------------
you     : find the notebooks


intent  : search_files  {'pattern': '*notebook*', 'workspace': 'active'}
speech  : I found 16 files matching *notebook*.
latency : ttft 152.1 ms · total 356.2 ms · 820 cached + 46 new tokens
----------------------------------------------------------------------
you     : which branch am I on


intent  : git_status  {}
speech  : You're on main with 15 changed files.
latency : ttft 224.9 ms · total 355.5 ms · 820 cached + 84 new tokens
----------------------------------------------------------------------
you     : show me AGENTS.md


intent  : read_file  {'path': 'AGENTS.md'}
speech  : Here's AGENTS.md, 37 lines shown.
latency : ttft 215.0 ms · total 414.5 ms · 820 cached + 111 new tokens
----------------------------------------------------------------------
you     : write me a haiku


intent  : unsupported  {'reason': 'Chit-chat is outside my developer scope'}
speech  : Sorry, that's outside what I can do.
latency : ttft 261.2 ms · total 486.0 ms · 820 cached + 117 new tokens
----------------------------------------------------------------------


## Confirmation flow

Writes and app launches need an explicit **yes**. The decision is held in the
session until you answer (here we cancel, so nothing is saved).

In [4]:
resp = r2.ask("remember that my favourite editor is Cursor")
print(resp.text, "| state:", resp.state)
print(r2.cancel().text)

Remember editor = Cursor? Say yes to confirm or no to cancel. | state: awaiting_confirmation
Okay, cancelled.


## Interactive chat
Type a request and press **Ask** (or Enter). Tick *speak* to hear R2.

In [5]:
import ipywidgets as W
from IPython.display import display, HTML
import html

box = W.Text(placeholder="e.g. how big is my diff?", layout=W.Layout(width="70%"))
ask_b = W.Button(description="Ask", button_style="primary")
yes_b = W.Button(description="Yes", button_style="success", disabled=True)
no_b = W.Button(description="No", button_style="danger", disabled=True)
speak = W.Checkbox(value=False, description="speak")
out = W.Output(layout=W.Layout(border="1px solid #ddd", padding="8px", max_height="420px",
                               overflow="auto"))

def render(user, resp):
    m = resp.metrics or {}
    color = "#d9a300" if resp.needs_confirmation else ("#0a8" if resp.tool_result and resp.tool_result.ok else "#c33")
    detail = "" if resp.text == resp.spoken_text() else f"<pre style='white-space:pre-wrap'>{html.escape(resp.text)}</pre>"
    meta = (f"{resp.decision.intent.value if resp.decision else '-'} · ttft {m.get('ttft_ms','-')} ms · "
            f"{m.get('cached_tokens','-')} cached tokens") if m else ""
    with out:
        display(HTML(f"<div><b>you</b>: {html.escape(user)}</div>"
                     f"<div style='border-left:4px solid {color};padding-left:8px;margin:4px 0 12px'>"
                     f"<b>R2</b>: {html.escape(resp.spoken_text())}{detail}"
                     f"<div style='color:#888;font-size:11px'>{meta}</div></div>"))
    yes_b.disabled = no_b.disabled = not resp.needs_confirmation
    if speak.value:
        r2.speak(resp.spoken_text(), block=False)

def on_ask(_=None):
    q = box.value.strip()
    if q:
        box.value = ""
        render(q, r2.ask(q))

ask_b.on_click(on_ask)
box.on_submit(on_ask) if hasattr(box, "on_submit") else box.observe(lambda c: None)
yes_b.on_click(lambda _: render("yes", r2.confirm()))
no_b.on_click(lambda _: render("no", r2.cancel()))
display(W.VBox([W.HBox([box, ask_b, speak]), W.HBox([yes_b, no_b]), out]))

/var/folders/hl/y1t66klj6vs22s295ltbldnm0000gn/T/ipykernel_94278/1224170464.py:35: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  box.on_submit(on_ask) if hasattr(box, "on_submit") else box.observe(lambda c: None)
